# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) and made available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Let's load the dataset metadata using the `mlcroissant` library. This will allow us to see an overview of the dataset contents, including available record sets and field IDs.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema (JSON-LD) URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset object
dataset = mlc.Dataset(croissant_url)

# Print metadata summary (name & description attributes are properties, not dict keys)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
We can now review available record sets and their fields. In Croissant, each *record set* represents a table or structured collection, where fields define columns or attributes. All are referenced by their `@id`.

Let's list every record set with its `@id`, label/name, and associated field `@id`s:

In [ ]:
# Gather record set information (referenced by their @id)
record_sets = dataset.metadata.record_sets
for recset in record_sets:
    print(f"Record Set @id: {recset.id}")
    print(f"  Name/Label: {recset.name or getattr(recset, 'label', '')}")
    print(f"  Fields:")
    for field in recset.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}")
    print('-' * 40)

## 3. Data Extraction
To analyze the data, we'll extract one or more record sets into pandas DataFrames (referenced by their `@id`). This enables efficient transformations and statistical summaries in later steps.

First, specify which record set(s) to pull, using their `@id`.

In [ ]:
# For this dataset, let's list all record set ids (from previous output)
record_set_ids = [recset.id for recset in dataset.metadata.record_sets]
print("Record sets available (by @id):", record_set_ids)

# Let's select the main proteins record set; commonly the main data table is the first
main_record_set_id = record_set_ids[0]  # Change index if needed based on output above

# Load the records from each record set and create DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {df.shape[0]} records from {record_set_id}")

# Show columns of main record set
print(f"Columns for Record Set {main_record_set_id} (@id):\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now, we'll demonstrate basic EDA: filtering records by a numeric field, normalization, and groupings. All field references are by their `@id`.

First, let's print all available field IDs in the main record set again so we can select one numeric field for demonstration.

In [ ]:
main_recset = [rs for rs in dataset.metadata.record_sets if rs.id == main_record_set_id][0]
print("Available fields and their @id in main record set:")
for fld in main_recset.fields:
    print(f"  - {fld.id}: {fld.name} (dataType: {fld.data_type})")

**Choose one numeric field for demonstration.** For most protein mass spec datasets these are fields like `cr:coverage`, `cr:peptide_count`, or `cr:mw` (molecular weight).

For illustration, let's use the coverage field (e.g., `cr:coverage`), and group by a categorical field (e.g., `cr:sample_id`, if present).

In [ ]:
# Set the @id of a representative numeric and category field
numeric_field_id = None
group_field_id = None

# Find candidate numeric fields (commonly Float/Integer)
for field in main_recset.fields:
    if field.data_type in ('schema:Float', 'schema:Integer', 'schema:Number'):
        if numeric_field_id is None:
            numeric_field_id = field.id
    if group_field_id is None and (field.data_type in ('schema:Text', 'schema:String')):
        group_field_id = field.id
print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

df = dataframes[main_record_set_id]

# Ensure correct types (some may be loaded as string)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove NaN
filtered_df = df[df[numeric_field_id].notnull()]

# Threshold filter (arbitrary: mean + 1 std if >10 is not sensible)
if filtered_df[numeric_field_id].max() < 10:
    threshold = filtered_df[numeric_field_id].mean()
    print(f"Setting threshold to mean: {threshold}")
else:
    threshold = 10

filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group and aggregate if possible
if group_field_id in filtered_df.columns and filtered_df[group_field_id].notnull().any():
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and (if grouping is available) show the mean per group.

You can customize the plots below for your fields of interest.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram
sns.histplot(filtered_df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping available, barplot of group means
if group_field_id in filtered_df.columns and filtered_df[group_field_id].notnull().any():
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    grouped.head(20).plot(kind='bar', figsize=(10,4))
    plt.title(f"Mean {numeric_field_id} per {group_field_id} (top 20)")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR² protein mass spectrometry dataset using `mlcroissant`.
- All dataset elements (record sets, fields) were referenced by their Croissant `@id`.
- We inspected available structured tables, loaded the main record set, and conducted basic EDA and visualizations.
- For advanced analysis, you might continue with differential analysis, downstream ML, or linking this data to external bioinformatic resources, always referencing fields and tables using their `@id` as shown above.

*Tip: Use `dataset.metadata.to_json()` for the full Croissant metadata, or examine the fields on `dataset.metadata.record_sets` for more details on available variables and their provenance.*